# Sleep-EDF Expanded — Cassette pipeline (7 channels + age, sex, stage)

**What this notebook builds** (from the `sleep-cassette` folder only):

| Output | Content |
|---|---|
| `recordings/<SC4xxxE0>.parquet` | sample-level table @100 Hz: `recording_key, subject_id, night, epoch_idx`, the 7 cassette channels, `age, sex, stage` |
| `epoch_table.parquet` | one row per kept 30-s epoch (metadata + label only, small) |
| `recording_qc.csv` | one row per recording with every QC counter |
| `run_config.json` | parameters, channel units, library versions |

**Decisions carried over from the discussion**

1. **Cassette only** (healthy subjects, no drug). Telemetry is *not* used: its second night per subject follows temazepam, and its EMG/hardware differ.
2. **Native sampling rates.** EEG/EOG are 100 Hz; Resp, EMG *envelope*, Temp and Event marker are **1 Hz**. MNE resamples the 1 Hz channels when a file mixes rates (see section 6), so the PSG is read with a small native EDF reader; slow channels are brought to 100 Hz by **sample-and-hold** (`np.repeat`), never interpolated.
3. **5 classes** `W, N1, N2, N3, REM`; R&K stages 3 and 4 are merged into N3; *Movement time* and *unscored* epochs are dropped and counted.
4. **Wake trimming** keeps at most `TRIM_MARGIN_MIN` minutes of Wake before the first and after the last sleep epoch. Wake *inside* the night (WASO) is never touched. Set `TRIM_MARGIN_MIN = None` to disable.
5. **Age / sex** come from `SC-subjects.xls`; the sex coding is read from the column header (never assumed) and cross-checked against the EDF header when the header is parseable.
6. `subject_id` and `night` are kept so that train/validation/test can be split **by subject** (both nights of a person must stay together).

Run the cells top to bottom. First run on a few recordings (`LIMIT_RECORDINGS = 6`), then set it to `None`.

In [1]:
import importlib.util

REQUIRED = ("numpy", "pandas", "pyarrow", "xlrd", "mne")
missing = [name for name in REQUIRED if importlib.util.find_spec(name) is None]
if missing:
    raise ImportError(f"Missing packages: {missing}\nInstall with:  pip install {' '.join(missing)}")
print("All required packages are available.")

All required packages are available.


In [2]:
import json
import os
import re
import time
import traceback
import warnings
from collections import Counter
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Sequence

import mne
import numpy as np
import pandas as pd

mne.set_log_level("ERROR")
warnings.filterwarnings("ignore", message=r".*(highpass|lowpass).*")  # harmless EDF prefilter warnings

# ======================= CONFIGURATION =======================
DATA_ROOT = Path(os.environ.get("SLEEP_EDF_ROOT", r"D:\Hackaton\sleep-edf-database-expanded-1.0.0"))
# Outputs go OUTSIDE the dataset folder so the original files (and SHA256SUMS.txt) stay untouched.
OUTPUT_ROOT = Path(os.environ.get("SLEEP_EDF_OUT", str(DATA_ROOT.parent / "sleep_edf_cassette_processed")))

TRIM_MARGIN_MIN = 30          # minutes of Wake kept before first / after last sleep epoch; None = keep all Wake
LIMIT_RECORDINGS = None       # e.g. 6 for a smoke test; None = all recordings
HEADER_MISMATCH = "raise"     # spreadsheet vs EDF-header disagreement: "raise" or "warn"
REQUIRE_METADATA = True       # every recording must have an age/sex row in SC-subjects.xls
SEX_CODING_OVERRIDE = None    # e.g. {1: "F", 2: "M"} to bypass automatic decoding of the spreadsheet
AGE_TOLERANCE_YEARS = 0       # allowed |spreadsheet age - EDF header age| before a mismatch is reported

# ======================= DATASET CONSTANTS =======================
CASSETTE_DIR = DATA_ROOT / "sleep-cassette"
SC_SUBJECTS_XLS = DATA_ROOT / "SC-subjects.xls"
RECORDINGS_DIR = OUTPUT_ROOT / "recordings"

FS_FAST_HZ = 100                                 # EEG, EOG
FS_SLOW_HZ = 1                                   # Resp, EMG envelope, Temp, Event marker
EPOCH_S = 30
SAMPLES_PER_EPOCH = EPOCH_S * FS_FAST_HZ         # 3000
HOLD_FACTOR = FS_FAST_HZ // FS_SLOW_HZ           # 100 samples per slow sample
TOL_S = 1e-6

FAST_CHANNELS = ("EEG Fpz-Cz", "EEG Pz-Oz", "EOG horizontal")
SLOW_CHANNELS = ("Resp oro-nasal", "EMG submental", "Temp rectal", "Event marker")
ALL_CHANNELS = FAST_CHANNELS + SLOW_CHANNELS     # names exactly as verified on the 153 real recordings

STAGE_BY_DESCRIPTION = {
    "Sleep stage W": "W",
    "Sleep stage 1": "N1",
    "Sleep stage 2": "N2",
    "Sleep stage 3": "N3",
    "Sleep stage 4": "N3",                       # R&K S3 + S4 -> AASM N3
    "Sleep stage R": "REM",
}
EXCLUDED_DESCRIPTIONS = {"Movement time": "movement", "Sleep stage ?": "unscored"}
STAGE_ORDER = ("W", "N1", "N2", "N3", "REM")
STAGE_ID = {stage: i for i, stage in enumerate(STAGE_ORDER)}
SLEEP_IDS = np.array([STAGE_ID[s] for s in ("N1", "N2", "N3", "REM")], dtype=np.int8)

MARGIN_EPOCHS = None if TRIM_MARGIN_MIN is None else int(TRIM_MARGIN_MIN * 60 / EPOCH_S)

print("DATA_ROOT  :", DATA_ROOT, "| exists:", DATA_ROOT.is_dir())
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("Trimming   :", "off" if MARGIN_EPOCHS is None else f"{TRIM_MARGIN_MIN} min = {MARGIN_EPOCHS} epochs")

DATA_ROOT  : D:\Hackaton\sleep-edf-database-expanded-1.0.0 | exists: True
OUTPUT_ROOT: D:\Hackaton\sleep_edf_cassette_processed
Trimming   : 30 min = 60 epochs


## 1. Directory inventory

Facts taken from the exploration notebook (real data): `sleep-cassette` holds 306 EDF files = 153 PSG + 153 Hypnogram; `sleep-telemetry` holds 88 = 44 + 44; the root also has `RECORDS`, `RECORDS-v1`, `SHA256SUMS.txt`, `SC-subjects.xls`, `ST-subjects.xls`.

In [3]:
if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f"DATA_ROOT does not exist: {DATA_ROOT}")

print("Main directory contents:\n")
for item in sorted(DATA_ROOT.iterdir()):
    print(f"{'DIR ' if item.is_dir() else 'FILE':5}: {item.name}")

edf_files = pd.DataFrame({"path": sorted(DATA_ROOT.rglob("*.edf"))})
edf_files["folder"] = edf_files["path"].map(lambda p: p.parent.name)
edf_files["kind"] = np.select(
    [edf_files["path"].map(lambda p: p.name.endswith("-PSG.edf")),
     edf_files["path"].map(lambda p: p.name.endswith("-Hypnogram.edf"))],
    ["PSG", "Hypnogram"], default="other")

counts = pd.crosstab(edf_files["folder"], edf_files["kind"])
print("\nEDF files by folder and kind:")
display(counts)

EXPECTED = {("sleep-cassette", "PSG"): 153, ("sleep-cassette", "Hypnogram"): 153}
for (folder, kind), expected in EXPECTED.items():
    found = int(counts.loc[folder, kind]) if folder in counts.index and kind in counts.columns else 0
    status = "OK " if found == expected else "WARN"
    print(f"[{status}] {folder}/{kind}: found {found}, expected {expected}")

for name in ("SC-subjects.xls",):
    print(f"[{'OK ' if (DATA_ROOT / name).is_file() else 'WARN'}] {name} present")

Main directory contents:

FILE : Data exploration new.ipynb
FILE : Data exploration.ipynb
FILE : RECORDS
FILE : RECORDS-v1
FILE : SC-subjects.xls
FILE : SHA256SUMS.txt
DIR  : sleep-cassette
DIR  : sleep-telemetry
FILE : SleepEDF_Cassette_Pipeline.ipynb
FILE : ST-subjects.xls

EDF files by folder and kind:


kind,Hypnogram,PSG
folder,,
sleep-cassette,153,153
sleep-telemetry,44,44


[OK ] sleep-cassette/PSG: found 153, expected 153
[OK ] sleep-cassette/Hypnogram: found 153, expected 153
[OK ] SC-subjects.xls present


## 2. Native EDF reader

`mne.io.read_raw_edf` resamples every channel to the highest rate in the file. For the 1 Hz channels that is a band-limited resampling, not a hold (demonstrated in section 6). This small reader follows the EDF specification directly:

* header = 256 bytes + 256 bytes per signal; data = int16 little-endian, record after record;
* physical value = `gain * digital + offset`, with `gain = (phys_max - phys_min) / (dig_max - dig_min)`;
* every channel is returned at its **own native rate**, in the **units stored in the file** (µV for EEG/EOG/EMG).

It is validated at run time against MNE on the first recording (section 6).

In [4]:
@dataclass(frozen=True)
class EdfHeader:
    patient: str
    recording: str
    start: datetime
    n_records: int
    record_s: float
    header_bytes: int
    labels: tuple
    units: tuple
    samples_per_record: tuple
    gain: tuple
    offset: tuple

    def rate_hz(self, label: str) -> float:
        return self.samples_per_record[self.labels.index(label)] / self.record_s


_SIGNAL_FIELDS = (("label", 16), ("transducer", 80), ("unit", 8), ("phys_min", 8), ("phys_max", 8),
                  ("dig_min", 8), ("dig_max", 8), ("prefilter", 80), ("n_samples", 8), ("reserved", 32))


def _text(raw: bytes) -> str:
    return raw.decode("ascii", errors="replace").strip()


def _number(text: str) -> float:
    """float() that tolerates blank/odd fields (seen in EDF+ annotation signals); such a signal is never read as data."""
    try:
        return float(text)
    except ValueError:
        return float("nan")


def read_edf_header(path: Path) -> EdfHeader:
    """Parse an EDF / EDF+ header (only the first 256 * (n_signals + 1) bytes are read)."""
    with path.open("rb") as fh:
        fixed = fh.read(256)
        if len(fixed) != 256 or fixed[:8] != b"0       ":
            raise ValueError(f"{path.name}: not an EDF/EDF+ file (bad version field)")
        n_sig = int(_text(fixed[252:256]))
        block = fh.read(256 * n_sig)
    if len(block) != 256 * n_sig:
        raise ValueError(f"{path.name}: truncated signal header")

    columns, pos = {}, 0
    for name, width in _SIGNAL_FIELDS:
        columns[name] = [_text(block[pos + i * width: pos + (i + 1) * width]) for i in range(n_sig)]
        pos += width * n_sig

    day, month, year = (int(x) for x in _text(fixed[168:176]).split("."))
    hour, minute, second = (int(x) for x in _text(fixed[176:184]).split("."))
    year += 1900 if year >= 85 else 2000                     # EDF clipping date rule

    header_bytes = int(_text(fixed[184:192]))
    if header_bytes != 256 * (n_sig + 1):
        raise ValueError(f"{path.name}: inconsistent header length {header_bytes} for {n_sig} signals")

    phys_min, phys_max = (np.array([_number(x) for x in columns[k]]) for k in ("phys_min", "phys_max"))
    dig_min, dig_max = (np.array([_number(x) for x in columns[k]]) for k in ("dig_min", "dig_max"))
    with np.errstate(divide="ignore", invalid="ignore"):
        gain = (phys_max - phys_min) / (dig_max - dig_min)
    offset = phys_min - gain * dig_min

    return EdfHeader(
        patient=_text(fixed[8:88]), recording=_text(fixed[88:168]),
        start=datetime(year, month, day, hour, minute, second),
        n_records=int(_text(fixed[236:244])), record_s=float(_text(fixed[244:252])),
        header_bytes=header_bytes, labels=tuple(columns["label"]), units=tuple(columns["unit"]),
        samples_per_record=tuple(int(x) for x in columns["n_samples"]),
        gain=tuple(gain), offset=tuple(offset))


def read_edf_signals(path: Path, header: EdfHeader, labels: Sequence[str]) -> dict:
    """Return {label: float32 array in the file's physical units, at the channel's native rate}."""
    spr = np.asarray(header.samples_per_record, dtype=np.int64)
    record_len = int(spr.sum())
    payload = path.stat().st_size - header.header_bytes
    n_records = header.n_records if header.n_records >= 0 else payload // (2 * record_len)
    if payload < 2 * record_len * n_records:
        raise ValueError(f"{path.name}: file is truncated ({payload} payload bytes for {n_records} records)")
    data = np.fromfile(path, dtype="<i2", count=n_records * record_len, offset=header.header_bytes)
    data = data.reshape(n_records, record_len)
    edges = np.concatenate(([0], np.cumsum(spr)))
    signals = {}
    for label in labels:
        i = header.labels.index(label)
        digital = data[:, edges[i]:edges[i + 1]].reshape(-1)
        signals[label] = (digital * header.gain[i] + header.offset[i]).astype(np.float32)
    return signals


def read_hypnogram(path: Path):
    """Hypnogram annotations as (onset_s, duration_s, description); MNE reader is verified on the real files."""
    ann = mne.read_annotations(path)
    # list(map(str, ...)) instead of asarray(dtype=str): newer NumPy/MNE store descriptions as StringDType
    return (np.asarray(ann.onset, dtype=float), np.asarray(ann.duration, dtype=float),
            np.array(list(map(str, ann.description))))

## 3. Pure helpers (labels, trimming, metadata) + self-tests

These functions do not touch the disk, so they are tested immediately below with tiny hand-made inputs.

In [5]:
@dataclass(frozen=True)
class EpochLabels:
    stage_id: np.ndarray    # int8, index into STAGE_ORDER, -1 = epoch not usable
    reason: np.ndarray      # object: ok | movement | unscored | unknown_annotation | no_annotation


def _annotation_reason(description: str) -> str:
    if description in STAGE_BY_DESCRIPTION:
        return "ok"
    return EXCLUDED_DESCRIPTIONS.get(description, "unknown_annotation")


def annotation_qc(start_s: np.ndarray, end_s: np.ndarray) -> dict:
    """Grid alignment, gaps and overlaps of the hypnogram timeline."""
    boundaries = np.concatenate([start_s, end_s]) / EPOCH_S
    order = np.argsort(start_s, kind="stable")
    gap = start_s[order][1:] - end_s[order][:-1]
    return {
        "off_grid": int(np.sum(~np.isclose(boundaries, np.round(boundaries), atol=TOL_S))),
        "gaps": int(np.sum(gap > TOL_S)),
        "overlaps": int(np.sum(gap < -TOL_S)),
    }


def label_epochs(start_s: np.ndarray, end_s: np.ndarray, description: np.ndarray, n_epochs: int) -> EpochLabels:
    """Assign each 30-s epoch [k*30, (k+1)*30) the annotation that covers its start (vectorised)."""
    if start_s.size == 0:
        raise ValueError("hypnogram has no annotations")
    order = np.argsort(start_s, kind="stable")
    start_s, end_s, description = start_s[order], end_s[order], description[order]
    if np.any(end_s[:-1] > start_s[1:] + TOL_S):
        raise ValueError("overlapping hypnogram annotations")

    ann_stage = np.array([STAGE_ID.get(STAGE_BY_DESCRIPTION.get(d), -1) for d in description], dtype=np.int8)
    ann_reason = np.array([_annotation_reason(d) for d in description], dtype=object)

    epoch_start = np.arange(n_epochs) * EPOCH_S
    idx = np.searchsorted(start_s, epoch_start + TOL_S, side="right") - 1      # last annotation starting <= epoch start
    inside = idx >= 0
    safe = np.where(inside, idx, 0)
    covered = inside & (epoch_start + TOL_S < end_s[safe])                    # ... and not yet finished

    stage_id = np.where(covered, ann_stage[safe], -1).astype(np.int8)
    reason = np.where(covered, ann_reason[safe], "no_annotation")
    return EpochLabels(stage_id=stage_id, reason=reason)


def sleep_bounds(stage_id: np.ndarray) -> tuple:
    """(first, last) epoch index whose stage is N1/N2/N3/REM."""
    sleep = np.flatnonzero(np.isin(stage_id, SLEEP_IDS))
    if sleep.size == 0:
        raise ValueError("recording contains no sleep epochs")
    return int(sleep[0]), int(sleep[-1])


def trim_window(stage_id: np.ndarray, margin_epochs) -> tuple:
    """Half-open epoch window [lo, hi): first/last sleep epoch +- margin. Wake inside the window (WASO) stays."""
    n = int(stage_id.size)
    if margin_epochs is None:
        return 0, n
    first, last = sleep_bounds(stage_id)
    return max(first - margin_epochs, 0), min(last + margin_epochs + 1, n)


# ---------------------------- spreadsheet / header helpers ----------------------------
def find_column(df: pd.DataFrame, pattern: str, what: str, required: bool = True):
    hits = [c for c in df.columns if re.search(pattern, str(c), flags=re.I)]
    if len(hits) == 1:
        return hits[0]
    if not hits and not required:
        return None
    raise KeyError(f"Cannot identify the '{what}' column. Columns: {list(df.columns)}; matches: {hits}")


def decode_sex(values: pd.Series, column_name: str, override=None) -> pd.Series:
    """Return 'F'/'M' (NaN kept). The numeric coding is taken from the column header, e.g. 'sex (F=1)'."""
    non_null = values.dropna()
    if len(non_null) and non_null.map(lambda v: isinstance(v, str)).all():
        letters = non_null.str.strip().str.upper().str[:1]
        if not set(letters) <= {"F", "M"}:
            raise ValueError(f"Unrecognised text values in sex column: {sorted(set(non_null))}")
        return values.str.strip().str.upper().str[:1]

    numeric = pd.to_numeric(values)
    codes = sorted({int(v) for v in numeric.dropna()})
    mapping = dict(override) if override else {int(n): l for l, n in re.findall(r"([FM])\s*=\s*(\d+)", column_name.upper())}
    if len(mapping) == 1 and len(codes) == 2:          # header names one letter only; the other code is the other sex
        (known_code, known_letter), = mapping.items()
        other_code = next(c for c in codes if c != known_code)
        mapping[other_code] = "M" if known_letter == "F" else "F"
        print(f"  (header gives only '{known_letter}={known_code}'; inferred {other_code} -> {mapping[other_code]})")
    if not codes or not set(codes) <= set(mapping):
        raise ValueError(f"Cannot decode sex codes {codes} from column '{column_name}' (parsed mapping: {mapping}). "
                         "Set SEX_CODING_OVERRIDE explicitly.")
    return numeric.map(lambda v: mapping[int(v)] if pd.notna(v) else pd.NA)


_SEX_TOKEN = re.compile(r"(?<![A-Za-z0-9])(FEMALE|MALE|F|M)(?![A-Za-z0-9])", re.I)
_DATE_LIKE = re.compile(r"\d{1,2}[-./][A-Za-z]{3}[-./]\d{2,4}|\d{4}-\d{2}-\d{2}|\d{1,2}[./]\d{1,2}[./]\d{2,4}")
_INT_TOKEN = re.compile(r"(?<!\d)(\d{1,3})(?!\d)")


def parse_patient_field(text: str) -> tuple:
    """Best-effort (sex, age) from the EDF 'local patient identification'. None = not parseable/ambiguous."""
    sexes = {tok.upper()[0] for tok in _SEX_TOKEN.findall(text)}
    sex = sexes.pop() if len(sexes) == 1 else None
    ints = [int(t) for t in _INT_TOKEN.findall(text) if 1 <= int(t) <= 120]
    age = ints[0] if len(ints) == 1 and not _DATE_LIKE.search(text) else None
    return sex, age


def _missing(value) -> bool:
    return value is None or bool(pd.isna(value))


def _constant_categorical(value, n: int, categories) -> pd.Categorical:
    code = -1 if _missing(value) else list(categories).index(value)
    return pd.Categorical.from_codes(np.full(n, code, dtype=np.int8), categories=list(categories))


def _constant_int16(value, n: int):
    missing = _missing(value)
    return pd.arrays.IntegerArray(np.full(n, 0 if missing else int(value), dtype=np.int16), np.full(n, missing))

In [6]:
# ---- label_epochs / trimming --------------------------------------------------
start = np.array([0.0, 90.0, 120.0, 150.0, 180.0, 210.0])
end   = np.array([90.0, 120.0, 150.0, 180.0, 210.0, 300.0])
desc  = np.array(["Sleep stage W", "Sleep stage 1", "Movement time", "Sleep stage 4", "Sleep stage R", "Sleep stage W"])
lab = label_epochs(start, end, desc, n_epochs=12)             # PSG is longer (360 s) than the hypnogram (300 s)
assert lab.stage_id.tolist() == [0, 0, 0, 1, -1, 3, 4, 0, 0, 0, -1, -1], lab.stage_id.tolist()
assert lab.reason.tolist()[4] == "movement" and lab.reason.tolist()[-1] == "no_annotation"
assert annotation_qc(start, end) == {"off_grid": 0, "gaps": 0, "overlaps": 0}

sid = np.array([0] * 10 + [1, 2, -1, 0, 0, 3, 4] + [0] * 10, dtype=np.int8)
assert sleep_bounds(sid) == (10, 16)
assert trim_window(sid, 3) == (7, 20)                          # 3 Wake epochs before/after; WASO (idx 13,14) is inside
assert trim_window(sid, 500) == (0, sid.size)                  # margin larger than the recording is clipped
assert trim_window(sid, None) == (0, sid.size)
try:
    trim_window(np.zeros(5, dtype=np.int8), 3); raise AssertionError("expected ValueError")
except ValueError:
    pass
try:
    label_epochs(np.array([0., 20.]), np.array([30., 50.]), np.array(["Sleep stage W"] * 2), 2); raise AssertionError
except ValueError:
    pass

# ---- sample-and-hold (slow channel -> 100 Hz) -----------------------------------
held = np.repeat(np.array([5.0, 7.0], dtype=np.float32), HOLD_FACTOR)
assert held.size == 2 * HOLD_FACTOR and held[:HOLD_FACTOR].tolist() == [5.0] * HOLD_FACTOR and held[HOLD_FACTOR] == 7.0

# ---- sex decoding: coding comes from the header ----------------------------------
s = pd.Series([1, 2, 2, 1, np.nan])
assert decode_sex(s, "sex (F=1)").tolist()[:4] == ["F", "M", "M", "F"]
assert decode_sex(s, "Sex (F=1, M=2)").tolist()[:4] == ["F", "M", "M", "F"]
assert decode_sex(s, "sex", override={1: "M", 2: "F"}).tolist()[:4] == ["M", "F", "F", "M"]
assert decode_sex(pd.Series(["Female", "male"]), "sex").tolist() == ["F", "M"]
try:
    decode_sex(s, "sex"); raise AssertionError("expected ValueError (coding unknown)")
except ValueError:
    pass

# ---- EDF patient-field parsing (tolerant; unparseable -> None, never a wrong guess) ----
assert parse_patient_field("X F X X") == ("F", None)
assert parse_patient_field("F 33yr") == ("F", 33)
assert parse_patient_field("Male, 33 yr") == ("M", 33)
assert parse_patient_field("X F 01-JAN-1930 X") == ("F", None)          # date present -> age not guessed
assert parse_patient_field("M12 X") == (None, 12)                       # 'M12' is a code, not a sex token
assert parse_patient_field("X X X X") == (None, None)
print("All helper self-tests passed.")

  (header gives only 'F=1'; inferred 2 -> M)
All helper self-tests passed.


## 4. Subject metadata (age, sex) from `SC-subjects.xls`

Column names and the sex coding are **printed and decoded from the file itself**; nothing is assumed except that the columns contain the words *subject*, *night* (optional), *age* and *sex*.

In [7]:
def load_subject_metadata(path: Path) -> pd.DataFrame:
    raw = pd.read_excel(path, engine="xlrd")
    print("Columns in", path.name, ":", list(raw.columns))
    display(raw.head())

    c_subject = find_column(raw, r"^\s*subject", "subject")
    c_night = find_column(raw, r"^\s*night", "night", required=False)
    c_age = find_column(raw, r"^\s*age", "age")
    c_sex = find_column(raw, r"^\s*(sex|gender)", "sex")
    raw = raw.dropna(subset=[c_subject])

    print(f"\nRaw unique values in '{c_sex}':", sorted(raw[c_sex].dropna().unique().tolist()))
    sex = decode_sex(raw[c_sex], str(c_sex), SEX_CODING_OVERRIDE)
    decoded = pd.DataFrame({"raw": raw[c_sex], "decoded": sex}).dropna().drop_duplicates().sort_values("raw")
    print("Decoded sex coding used:"); display(decoded)

    age = pd.to_numeric(raw[c_age], errors="raise")
    if not (age.dropna() % 1 == 0).all():
        raise ValueError(f"Non-integer ages found in column '{c_age}'")

    meta = pd.DataFrame({
        "subject": pd.to_numeric(raw[c_subject], errors="raise").astype("int64"),
        "night": pd.to_numeric(raw[c_night], errors="raise").astype("Int64") if c_night else pd.array([pd.NA] * len(raw), dtype="Int64"),
        "age": age.astype("Int16"),
        "sex": sex,
    }).reset_index(drop=True)
    keys = ["subject", "night"] if c_night else ["subject"]
    if meta.duplicated(keys).any():
        raise ValueError(f"Duplicate {keys} rows in the spreadsheet:\n{meta[meta.duplicated(keys, keep=False)]}")
    return meta


subject_meta = load_subject_metadata(SC_SUBJECTS_XLS)
print(f"\n{len(subject_meta)} metadata rows; night column present: {subject_meta['night'].notna().any()}")
display(subject_meta.describe(include="all"))

Columns in SC-subjects.xls : ['subject', 'night', 'age', 'sex (F=1)', 'LightsOff']


,subject,night,age,sex (F=1),LightsOff
0,0,1,33,1,00:38:00
1,0,2,33,1,21:57:00
2,1,1,33,1,22:44:00
3,1,2,33,1,22:15:00
4,2,1,26,1,22:50:00



Raw unique values in 'sex (F=1)': [1, 2]
  (header gives only 'F=1'; inferred 2 -> M)
Decoded sex coding used:


,raw,decoded
0,1,F
20,2,M



153 metadata rows; night column present: True


,subject,night,age,sex
count,153.000000,153.0,153.0,153
unique,NaN,<NA>,<NA>,2
top,NaN,<NA>,<NA>,F
freq,NaN,<NA>,<NA>,82
mean,39.470588,1.503268,58.986928,NaN
std,23.694342,0.501631,22.1181,NaN
min,0.000000,1.0,25.0,NaN
25%,19.000000,1.0,34.0,NaN
50%,40.000000,2.0,57.0,NaN
75%,59.000000,2.0,73.0,NaN


## 5. Recording index: PSG ↔ Hypnogram pairing, subject/night, metadata join, header cross-check

* Pairing key = first 7 characters of the file name (`SC4001E`); the hypnogram's 8th letter is the scorer and differs from the PSG's `0`.
* `SC4ssNE…` → subject `ss`, night `N`. Not every subject has two nights (real data: subjects 13, 36, 52 have one), so the code never assumes a pair.

In [8]:
_KEY_RE = re.compile(r"^SC4(?P<subject>\d{2})(?P<night>\d)[A-Z]$")


@dataclass(frozen=True)
class Recording:
    name: str            # SC4001E0
    key: str             # SC4001E  (subject + night)
    psg: Path
    hypnogram: Path
    subject: int
    night: int

    @property
    def subject_id(self) -> str:
        return f"SC4{self.subject:02d}"      # same convention as the exploration notebook (SC400, SC401, ...)


def _files_by_key(files) -> dict:
    out = {}
    for path in files:
        key = path.name[:7]
        if key in out:
            raise RuntimeError(f"Duplicate key {key}: {out[key].name} and {path.name}")
        out[key] = path
    return out


def index_recordings(folder: Path) -> list:
    psg = _files_by_key(sorted(folder.glob("*-PSG.edf")))
    hyp = _files_by_key(sorted(folder.glob("*-Hypnogram.edf")))
    if set(psg) != set(hyp):
        raise RuntimeError(f"PSG without hypnogram: {sorted(set(psg) - set(hyp))}; hypnogram without PSG: {sorted(set(hyp) - set(psg))}")
    recordings = []
    for key in sorted(psg):
        match = _KEY_RE.match(key)
        if match is None:
            raise ValueError(f"Unexpected cassette file key: {key}")
        recordings.append(Recording(name=psg[key].name.removesuffix("-PSG.edf"), key=key, psg=psg[key], hypnogram=hyp[key],
                                    subject=int(match["subject"]), night=int(match["night"])))
    return recordings


all_recordings = index_recordings(CASSETTE_DIR)
recordings = all_recordings[:LIMIT_RECORDINGS] if LIMIT_RECORDINGS else all_recordings

index_df = pd.DataFrame([{"key": r.key, "name": r.name, "subject": r.subject, "subject_id": r.subject_id, "night": r.night}
                         for r in recordings])
print(f"Recordings indexed: {len(all_recordings)} (processing {len(recordings)})")
print(f"Unique subjects   : {index_df['subject'].nunique()}")
per_subject = index_df.groupby("subject").size()
print("Recordings/subject:", per_subject.value_counts().sort_index().to_dict())
single = per_subject[per_subject == 1].index.tolist()
print("Subjects with a single night:", [f"{s:02d} (night {int(index_df.loc[index_df.subject == s, 'night'].iloc[0])})" for s in single])

# ---- join metadata on (subject, night) -----------------------------------------------------------
join_keys = ["subject", "night"] if subject_meta["night"].notna().any() else ["subject"]
meta_for_join = subject_meta.copy()
if join_keys == ["subject"]:
    meta_for_join = meta_for_join.drop(columns="night")
index_df["night"] = index_df["night"].astype("int64")
meta_for_join = meta_for_join.astype({k: "int64" for k in join_keys})
recordings_df = index_df.merge(meta_for_join, on=join_keys, how="left", validate="many_to_one")

no_meta = recordings_df[recordings_df["age"].isna() | recordings_df["sex"].isna()]
if len(no_meta):
    msg = f"{len(no_meta)} recordings lack age/sex in {SC_SUBJECTS_XLS.name}:\n{no_meta[['name', 'subject', 'night']].to_string(index=False)}"
    if REQUIRE_METADATA:
        raise ValueError(msg)
    print("WARNING:", msg)
unused = subject_meta.merge(index_df, on=join_keys, how="left", indicator=True).query("_merge == 'left_only'")
print(f"Spreadsheet rows without a recording (ignored): {len(unused)}")

Recordings indexed: 153 (processing 153)
Unique subjects   : 78
Recordings/subject: {1: 3, 2: 75}
Subjects with a single night: ['13 (night 1)', '36 (night 2)', '52 (night 2)']
Spreadsheet rows without a recording (ignored): 0


In [9]:
# ---- read every PSG header (256+ bytes each) -> channel/rate/unit audit + demographics cross-check ----
header_rows, demog_rows = [], []
for rec in recordings:
    h = read_edf_header(rec.psg)
    for label, unit, spr in zip(h.labels, h.units, h.samples_per_record):
        header_rows.append({"name": rec.name, "channel": label, "unit": unit, "rate_hz": spr / h.record_s})
    sex_h, age_h = parse_patient_field(h.patient)
    demog_rows.append({"name": rec.name, "header_patient": h.patient, "header_sex": sex_h, "header_age": age_h})

header_df = pd.DataFrame(header_rows)
by_channel = header_df.groupby("channel")
channel_audit = pd.DataFrame({
    "n_recordings": by_channel["name"].nunique(),
    "units": by_channel["unit"].apply(lambda s: sorted(set(s))),
    "rates_hz": by_channel["rate_hz"].apply(lambda s: sorted(set(s))),
})
print("Channel audit across all indexed PSG files:")
display(channel_audit)

unexpected = channel_audit.index.difference(list(ALL_CHANNELS))
if len(unexpected):
    print("WARNING: channels not in ALL_CHANNELS (ignored):", list(unexpected))
for channel in ALL_CHANNELS:
    expected_rate = FS_FAST_HZ if channel in FAST_CHANNELS else FS_SLOW_HZ
    if channel not in channel_audit.index:
        print(f"WARNING: channel '{channel}' is absent from every file")
    elif channel_audit.loc[channel, "rates_hz"] != [float(expected_rate)]:
        raise ValueError(f"'{channel}': sampling rates {channel_audit.loc[channel, 'rates_hz']}, expected {expected_rate} Hz")
CHANNEL_UNITS = {c: channel_audit.loc[c, "units"] for c in ALL_CHANNELS if c in channel_audit.index}
print("Channel counts per recording (7 expected):", header_df.groupby("name").size().value_counts().to_dict())

print("\nRaw EDF patient fields of the first 5 recordings (verify the format):")
demog_df = pd.DataFrame(demog_rows)
display(demog_df.head())

check = recordings_df.merge(demog_df, on="name")
sex_bad = check[check["header_sex"].notna() & (check["header_sex"] != check["sex"])]
age_bad = check[check["header_age"].notna() & ((check["header_age"].astype("float64") - check["age"].astype("float64")).abs() > AGE_TOLERANCE_YEARS)]
print(f"Header sex parseable in {check['header_sex'].notna().sum()}/{len(check)} files; age in {check['header_age'].notna().sum()}/{len(check)}")
if len(sex_bad) or len(age_bad):
    msg = (f"Spreadsheet vs EDF-header disagreement:\n{sex_bad[['name', 'sex', 'header_sex']].to_string(index=False)}\n"
           f"{age_bad[['name', 'age', 'header_age']].to_string(index=False)}")
    if HEADER_MISMATCH == "raise":
        raise ValueError(msg)
    print("WARNING:", msg)
else:
    print("Spreadsheet age/sex agree with every parseable EDF header.")

Channel audit across all indexed PSG files:


,n_recordings,units,rates_hz
channel,,,
EEG Fpz-Cz,153,[uV],[100.0]
EEG Pz-Oz,153,[uV],[100.0]
EMG submental,153,[uV],[1.0]
EOG horizontal,153,[uV],[100.0]
Event marker,153,[],[1.0]
Resp oro-nasal,153,[],[1.0]
Temp rectal,153,"[, DegC]",[1.0]


Channel counts per recording (7 expected): {7: 153}

Raw EDF patient fields of the first 5 recordings (verify the format):


,name,header_patient,header_sex,header_age
0,SC4001E0,X F X Female_33yr,F,33
1,SC4002E0,X F X Female_33yr,F,33
2,SC4011E0,X F X Female_33yr,F,33
3,SC4012E0,X F X Female_33yr,F,33
4,SC4021E0,X F X Female_26yr,F,26


Header sex parseable in 153/153 files; age in 153/153


ValueError: Spreadsheet vs EDF-header disagreement:
    name sex header_sex
SC4062E0   F          M
SC4231E0   F          M
SC4742E0   M          F
    name  age  header_age
SC4231E0   50          49
SC4232E0   50          49
SC4481F0   67          66
SC4482F0   67          66
SC4531E0   67          66
SC4532E0   67          66
SC4642E0   85          88
SC4772G0   85         100

## 6. Validate the reader on the first recording (MNE cross-check + why MNE is not used for 1 Hz channels)

In [ ]:
first = recordings[0]
psg_header = read_edf_header(first.psg)
print(f"{first.name}: start {psg_header.start}, {psg_header.n_records} records x {psg_header.record_s:.0f} s")

# (a) exact agreement with MNE on the 100 Hz EEG channel (MNE returns volts, the reader returns the file's uV)
mine = read_edf_signals(first.psg, psg_header, ["EEG Fpz-Cz"])["EEG Fpz-Cz"]
raw_fast = mne.io.read_raw_edf(first.psg, include=["EEG Fpz-Cz"], preload=False, verbose="error")
n_check = min(raw_fast.n_times, 30 * 60 * FS_FAST_HZ)
mne_uv = raw_fast.get_data(start=0, stop=n_check)[0] * 1e6
assert raw_fast.n_times == mine.size, (raw_fast.n_times, mine.size)
assert np.max(np.abs(mne_uv - mine[:n_check])) < 1e-3, "reader disagrees with MNE on EEG Fpz-Cz"
assert raw_fast.info["meas_date"].replace(tzinfo=None) == psg_header.start, "start time differs from MNE"
print(f"[OK] EEG Fpz-Cz identical to MNE (max |diff| = {np.max(np.abs(mne_uv - mine[:n_check])):.2e} uV), n_samples and start time match")

# (b) how much does MNE's default mixed-rate reading distort the 1 Hz channels?
slow_present = [c for c in ("EMG submental", "Event marker") if c in psg_header.labels]
native = read_edf_signals(first.psg, psg_header, slow_present)
raw_all = mne.io.read_raw_edf(first.psg, preload=False, verbose="error")
n_slow = 20 * 60                                                         # first 20 minutes, in seconds
rows = []
for channel in slow_present:
    held = np.repeat(native[channel][:n_slow].astype(float), HOLD_FACTOR)   # what the sample-and-hold table contains
    via_mne = raw_all.get_data(picks=[channel], start=0, stop=n_slow * FS_FAST_HZ)[0]
    scale = float(held @ via_mne / (held @ held)) if held.any() else 1.0    # unit-free comparison (uV vs V)
    residual = np.linalg.norm(via_mne - scale * held) / max(np.linalg.norm(scale * held), 1e-12)
    rows.append({"channel": channel, "MNE default vs hold (relative residual)": round(float(residual), 4)})
print("\nDeviation of MNE's default 100 Hz version of the slow channels from the true 1 Hz samples held constant:")
display(pd.DataFrame(rows))
print("A residual above 0 means MNE inserted interpolated values between the recorded 1 Hz samples (with over/undershoot at steps);\n"
      "the table built here keeps only the recorded samples, each held for one second.")

## 7. Build the per-recording tables

`process_recording` returns (a) the sample-level frame for the kept epochs and (b) a small epoch table plus a QC dictionary. Invariants are asserted: every epoch has exactly 3000 rows, rows are epoch-contiguous, and the slow channels are constant inside each second.

In [ ]:
@dataclass(frozen=True)
class RecordingResult:
    frame: pd.DataFrame      # sample-level table, kept epochs only
    epochs: pd.DataFrame     # one row per kept epoch
    qc: dict


def process_recording(rec: Recording, age, sex, margin_epochs=MARGIN_EPOCHS) -> RecordingResult:
    # ---------- PSG: native channels ----------
    header = read_edf_header(rec.psg)
    present = [c for c in ALL_CHANNELS if c in header.labels]
    missing = [c for c in ALL_CHANNELS if c not in header.labels]
    for channel in present:
        expected = FS_FAST_HZ if channel in FAST_CHANNELS else FS_SLOW_HZ
        if not np.isclose(header.rate_hz(channel), expected):
            raise ValueError(f"{rec.name}: '{channel}' is sampled at {header.rate_hz(channel)} Hz, expected {expected} Hz")
    signals = read_edf_signals(rec.psg, header, present)

    per_epoch = {c: (SAMPLES_PER_EPOCH if c in FAST_CHANNELS else EPOCH_S * FS_SLOW_HZ) for c in present}
    n_epochs = min(len(signals[c]) // per_epoch[c] for c in present)
    tail_samples = int(sum(len(signals[c]) - n_epochs * per_epoch[c] for c in present))

    # ---------- hypnogram -> epoch labels ----------
    onset, duration, description = read_hypnogram(rec.hypnogram)
    hyp_header = read_edf_header(rec.hypnogram)
    start_offset_s = (hyp_header.start - header.start).total_seconds()      # 0 in a consistent dataset
    onset = onset + start_offset_s
    end = onset + duration
    ann = annotation_qc(onset, end)
    labels = label_epochs(onset, end, description, n_epochs)
    stage_id = labels.stage_id
    unknown = sorted({d for d in description if d not in STAGE_BY_DESCRIPTION and d not in EXCLUDED_DESCRIPTIONS})

    # ---------- trimming (WASO is never removed) ----------
    lo, hi = trim_window(stage_id, margin_epochs)
    first_sleep, last_sleep = sleep_bounds(stage_id)
    in_window = (np.arange(n_epochs) >= lo) & (np.arange(n_epochs) < hi)
    keep = (stage_id >= 0) & in_window
    kept = np.flatnonzero(keep)
    n_kept = int(kept.size)
    n_rows = n_kept * SAMPLES_PER_EPOCH

    # ---------- sample-level frame ----------
    columns = {
        "recording_key": _constant_categorical(rec.key, n_rows, [rec.key]),
        "subject_id": _constant_categorical(rec.subject_id, n_rows, [rec.subject_id]),
        "night": np.full(n_rows, rec.night, dtype=np.int8),
        "epoch_idx": np.repeat(kept.astype(np.int32), SAMPLES_PER_EPOCH),
    }
    for channel in ALL_CHANNELS:
        if channel in signals:
            block = signals[channel][: n_epochs * per_epoch[channel]].reshape(n_epochs, per_epoch[channel])[kept].reshape(-1)
            if channel in SLOW_CHANNELS:
                block = np.repeat(block, HOLD_FACTOR)                       # sample-and-hold, no interpolation
        else:
            block = np.full(n_rows, np.nan, dtype=np.float32)               # missing channel -> NaN column (reported in QC)
        columns[channel] = block.astype(np.float32, copy=False)
    columns["age"] = _constant_int16(age, n_rows)
    columns["sex"] = _constant_categorical(sex, n_rows, ["F", "M"])
    columns["stage"] = pd.Categorical.from_codes(np.repeat(stage_id[kept], SAMPLES_PER_EPOCH), categories=list(STAGE_ORDER))
    frame = pd.DataFrame(columns)

    # ---------- invariants ----------
    assert len(frame) == n_rows
    assert (frame["epoch_idx"].value_counts() == SAMPLES_PER_EPOCH).all(), "an epoch does not have exactly 3000 rows"
    assert frame["epoch_idx"].is_monotonic_increasing
    for channel in SLOW_CHANNELS:
        if channel in signals and n_kept:
            per_second = frame[channel].to_numpy().reshape(-1, HOLD_FACTOR)
            assert np.array_equal(per_second, np.repeat(per_second[:, :1], HOLD_FACTOR, axis=1), equal_nan=True), f"{channel} not held per second"

    epochs = pd.DataFrame({
        "recording_key": rec.key, "subject_id": rec.subject_id, "subject": rec.subject, "night": rec.night,
        "epoch_idx": kept, "start_sec": kept * EPOCH_S, "stage_id": stage_id[kept],
        "age": _constant_int16(age, n_kept), "sex": _constant_categorical(sex, n_kept, ["F", "M"]),
    })
    epochs["stage"] = pd.Categorical.from_codes(epochs["stage_id"], categories=list(STAGE_ORDER))

    # ---------- QC ----------
    before = np.bincount(stage_id[stage_id >= 0], minlength=len(STAGE_ORDER))
    after = np.bincount(stage_id[kept], minlength=len(STAGE_ORDER))
    reasons = Counter(labels.reason.tolist())
    wake = stage_id == STAGE_ID["W"]
    idx = np.arange(n_epochs)
    qc = {
        "recording_key": rec.key, "name": rec.name, "subject_id": rec.subject_id, "night": rec.night,
        "age": None if _missing(age) else int(age), "sex": None if _missing(sex) else sex,
        "psg_hours": round(n_epochs * EPOCH_S / 3600, 3), "hyp_hours": round(float(end.max()) / 3600, 3),
        "hyp_start_offset_s": start_offset_s, "tail_samples_dropped": tail_samples,
        "missing_channels": ",".join(missing), "unknown_descriptions": ",".join(unknown),
        "n_annotations": int(onset.size), "off_grid": ann["off_grid"], "gaps": ann["gaps"], "overlaps": ann["overlaps"],
        "n_epochs": int(n_epochs), "n_valid": int((stage_id >= 0).sum()),
        "n_movement": reasons.get("movement", 0), "n_unscored": reasons.get("unscored", 0),
        "n_no_annotation": reasons.get("no_annotation", 0), "n_unknown_annotation": reasons.get("unknown_annotation", 0),
        "first_sleep_epoch": first_sleep, "last_sleep_epoch": last_sleep, "window_lo": lo, "window_hi": hi,
        "wake_dropped_leading": int((wake & (idx < lo)).sum()), "wake_dropped_trailing": int((wake & (idx >= hi)).sum()),
        "waso_kept": int((wake & (idx >= first_sleep) & (idx <= last_sleep)).sum()),
        "n_kept": n_kept, "n_rows": n_rows,
        **{f"pre_{s}": int(v) for s, v in zip(STAGE_ORDER, before)},
        **{f"kept_{s}": int(v) for s, v in zip(STAGE_ORDER, after)},
    }
    return RecordingResult(frame=frame, epochs=epochs, qc=qc)


demographics = recordings_df.set_index("key")[["age", "sex"]].to_dict("index")
print("process_recording defined; demographics available for", len(demographics), "recordings")

In [ ]:
smoke = recordings[0]
t0 = time.perf_counter()
result = process_recording(smoke, demographics[smoke.key]["age"], demographics[smoke.key]["sex"])
print(f"{smoke.name}: {len(result.frame):,} rows from {result.qc['n_kept']} kept epochs in {time.perf_counter() - t0:.1f} s "
      f"({result.frame.memory_usage(deep=True).sum() / 2**20:.0f} MiB in memory)")
display(result.frame.head(3))
print(result.frame.dtypes.to_string())
print("\nStage counts before -> after trimming:")
display(pd.DataFrame({"before": {s: result.qc[f"pre_{s}"] for s in STAGE_ORDER},
                      "after": {s: result.qc[f"kept_{s}"] for s in STAGE_ORDER}}))
print("Wake dropped (leading / trailing):", result.qc["wake_dropped_leading"], "/", result.qc["wake_dropped_trailing"],
      "| WASO epochs kept:", result.qc["waso_kept"])

## 8. Process all recordings and write the outputs

In [ ]:
RECORDINGS_DIR.mkdir(parents=True, exist_ok=True)

qc_rows, epoch_frames, failures = [], [], {}
for i, rec in enumerate(recordings, start=1):
    t0 = time.perf_counter()
    try:
        info = demographics[rec.key]
        res = process_recording(rec, info["age"], info["sex"])
        target = RECORDINGS_DIR / f"{rec.name}.parquet"
        res.frame.to_parquet(target, index=False, compression="zstd")
        res.qc["parquet_mb"] = round(target.stat().st_size / 2**20, 2)
        qc_rows.append(res.qc)
        epoch_frames.append(res.epochs)
        print(f"[{i:3d}/{len(recordings)}] {rec.name}  epochs {res.qc['n_epochs']:5d} -> kept {res.qc['n_kept']:5d}  "
              f"{res.qc['parquet_mb']:7.1f} MB  {time.perf_counter() - t0:5.1f} s")
    except Exception as exc:                       # keep going, report every failure at the end
        failures[rec.name] = f"{type(exc).__name__}: {exc}"
        print(f"[{i:3d}/{len(recordings)}] {rec.name}  FAILED -> {failures[rec.name]}")
        print(traceback.format_exc(limit=-3))

qc_df = pd.DataFrame(qc_rows)
epoch_table = pd.concat(epoch_frames, ignore_index=True)
qc_df.to_csv(OUTPUT_ROOT / "recording_qc.csv", index=False)
epoch_table.to_parquet(OUTPUT_ROOT / "epoch_table.parquet", index=False)
(OUTPUT_ROOT / "run_config.json").write_text(json.dumps({
    "trim_margin_min": TRIM_MARGIN_MIN, "epoch_s": EPOCH_S, "fs_fast_hz": FS_FAST_HZ, "fs_slow_hz": FS_SLOW_HZ,
    "channels": list(ALL_CHANNELS), "channel_units": CHANNEL_UNITS, "stage_order": list(STAGE_ORDER),
    "stage_by_description": STAGE_BY_DESCRIPTION, "n_recordings_ok": len(qc_rows), "failures": failures,
    "versions": {"numpy": np.__version__, "pandas": pd.__version__, "mne": mne.__version__},
}, indent=2, default=str), encoding="utf-8")
print(f"\nDone: {len(qc_rows)} ok, {len(failures)} failed. Outputs in {OUTPUT_ROOT}")
assert not failures, f"{len(failures)} recordings failed: {failures}"

## 9. Verification report

In [ ]:
print("=" * 70, "\nOVERVIEW\n", "=" * 70, sep="")
print(f"Recordings processed : {len(qc_df)}   subjects: {qc_df['subject_id'].nunique()}")
print(f"Epochs valid / kept  : {int(qc_df['n_valid'].sum()):,} / {int(qc_df['n_kept'].sum()):,}")
print(f"Rows written         : {int(qc_df['n_rows'].sum()):,}   parquet size: {qc_df['parquet_mb'].sum() / 1024:.2f} GB")

pre = qc_df[[f"pre_{s}" for s in STAGE_ORDER]].sum().rename(lambda c: c[4:])
kept = qc_df[[f"kept_{s}" for s in STAGE_ORDER]].sum().rename(lambda c: c[5:])
distribution = pd.DataFrame({"before_trim": pre, "before_%": (100 * pre / pre.sum()).round(1),
                             "kept": kept, "kept_%": (100 * kept / kept.sum()).round(1)})
print("\nClass distribution (all recordings):")
display(distribution)

print("Epochs excluded (per reason) and other flags:")
flags = pd.Series({
    "movement time": qc_df["n_movement"].sum(), "unscored (?)": qc_df["n_unscored"].sum(),
    "no annotation (PSG longer than hypnogram)": qc_df["n_no_annotation"].sum(),
    "unknown annotation text": qc_df["n_unknown_annotation"].sum(),
    "Wake dropped: leading": qc_df["wake_dropped_leading"].sum(), "Wake dropped: trailing": qc_df["wake_dropped_trailing"].sum(),
    "WASO epochs kept": qc_df["waso_kept"].sum(),
    "recordings with missing channels": int((qc_df["missing_channels"] != "").sum()),
    "recordings with hypnogram start offset != 0": int((qc_df["hyp_start_offset_s"].abs() > 1).sum()),
    "off-grid / gaps / overlaps": int(qc_df[["off_grid", "gaps", "overlaps"]].to_numpy().sum()),
})
display(flags.to_frame("count"))

print("Per-recording QC:")
display(qc_df[["name", "subject_id", "night", "age", "sex", "psg_hours", "hyp_hours", "n_epochs", "n_kept", "n_movement", "n_no_annotation",
               "missing_channels"] + [f"kept_{s}" for s in STAGE_ORDER]])

# Regression against numbers printed by the exploration notebook on the REAL files (untrimmed valid epochs)
KNOWN_FROM_EXPLORATION = {
    "SC4001E": {"n_epochs": 2650, "W": 1997, "N1": 58, "N2": 250, "N3": 220, "REM": 125},
    "SC4002E": {"n_epochs": 2830, "W": 1885, "N1": 59, "N2": 373, "N3": 297, "REM": 215},
}
for key, expected in KNOWN_FROM_EXPLORATION.items():
    row = qc_df[qc_df["recording_key"] == key]
    if row.empty:
        continue
    got = {"n_epochs": int(row["n_epochs"].iloc[0]), **{s: int(row[f"pre_{s}"].iloc[0]) for s in STAGE_ORDER}}
    if got == expected:
        print(f"[OK  ] {key}: identical to the counts printed by the exploration notebook")
    else:
        print(f"[DIFF] {key}: got {got}, exploration notebook had {expected}")

## 10. Using the outputs: arrays for a network + subject-wise split

In [ ]:
def load_recording_arrays(name: str):
    """X: (n_epochs, 7, 3000) float32, y: (n_epochs,) int class ids, epoch_idx: (n_epochs,) position in the night."""
    frame = pd.read_parquet(RECORDINGS_DIR / f"{name}.parquet")
    n = len(frame) // SAMPLES_PER_EPOCH
    epoch_idx = frame["epoch_idx"].to_numpy().reshape(n, SAMPLES_PER_EPOCH)
    assert (epoch_idx == epoch_idx[:, :1]).all(), "rows are not epoch-contiguous"
    X = (frame.loc[:, list(ALL_CHANNELS)].to_numpy(np.float32)
         .reshape(n, SAMPLES_PER_EPOCH, len(ALL_CHANNELS)).transpose(0, 2, 1))
    y = pd.Categorical(frame["stage"], categories=list(STAGE_ORDER)).codes.reshape(n, SAMPLES_PER_EPOCH)[:, 0]
    return X, y, epoch_idx[:, 0]


X, y, epoch_idx = load_recording_arrays(recordings[0].name)
print("X", X.shape, X.dtype, "| y", y.shape, np.bincount(y, minlength=len(STAGE_ORDER)).tolist(), "| channels:", ALL_CHANNELS)
expected_labels = epoch_table.loc[epoch_table["recording_key"] == recordings[0].key, "stage_id"].to_numpy()
assert np.array_equal(y, expected_labels), "labels in parquet differ from epoch_table"


def split_subjects(subject_ids: pd.Series, fractions=(0.7, 0.15, 0.15), seed: int = 42) -> dict:
    """Assign whole SUBJECTS (not epochs, not nights) to train/val/test."""
    subjects = np.array(sorted(subject_ids.unique()))
    np.random.default_rng(seed).shuffle(subjects)
    n_train = int(round(fractions[0] * len(subjects)))
    n_val = int(round(fractions[1] * len(subjects)))
    return {"train": subjects[:n_train].tolist(), "val": subjects[n_train:n_train + n_val].tolist(), "test": subjects[n_train + n_val:].tolist()}


split = split_subjects(epoch_table["subject_id"])
assert not (set(split["train"]) & set(split["val"])) and not (set(split["train"]) & set(split["test"])) and not (set(split["val"]) & set(split["test"]))
split_of = {s: name for name, members in split.items() for s in members}
assert epoch_table.groupby("subject_id")["subject_id"].apply(lambda s: s.map(split_of).nunique()).eq(1).all(), "a subject spans two splits"

summary = (epoch_table.assign(split=epoch_table["subject_id"].map(split_of))
           .pivot_table(index="split", columns="stage", values="epoch_idx", aggfunc="count", observed=True, fill_value=0)
           .reindex(columns=list(STAGE_ORDER)))
print("Subjects per split:", {k: len(v) for k, v in split.items()})
display(summary)